In [50]:
from pyspark.sql import SparkSession
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
import math as m
from pyspark.sql.window import Window
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from functools import reduce
from pyspark.ml.evaluation import RegressionEvaluator


In [51]:
spark = (
    SparkSession.builder.appName("Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

In [52]:
spark.sparkContext.setLogLevel("ERROR")
import logging
logging.getLogger('org.apache.spark.sql.execution.window.WindowExec').setLevel(logging.ERROR)

In [53]:
def create_pl(lags, time_length, type):

    """Creates a pipeline for model creation"""

    categorical_cols = ["biz_tags","rev_band"]
    indexer = StringIndexer(inputCols=categorical_cols, outputCols=[f"{c}_idx" for c in categorical_cols], handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols],
                        outputCols=[f"{c}_vec" for c in categorical_cols])
    
    feature_cols = (
        ["revenue"] +
        [f"rev_lag{l}" for l in lags] +
        [f"rev_mean_{w}" for w in [3,6,12]] +
        [f"rev_std_{w}" for w in [3,6,12]] +
        ["sin_month", "cos_month"] +
        [f"{c}_vec" for c in categorical_cols]
        )

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
    gbt = GBTRegressor(labelCol=f"{type}_{time_length}m", featuresCol="features", maxIter=100)   
    pipeline = Pipeline(stages=[indexer, encoder, assembler, gbt])
    return pipeline

def find_NULL(dfs):

    """Finds entries with NULL values over multiple datasets"""

    for df in dfs:
        condition = f.lit(False)
        
        condition = condition | f.col('rev_future_6m').isNull() & f.col('rev_future_12m').isNotNull()

        df.filter(condition).show()
    return df.filter(condition).show()

def impute_rev_lags_by_business(data, lags, group_col, order_col='year_month'):
    """
    Imputes missing revenue lag columns (rev_lag1, rev_lag3, etc.)
    with the mean of that lag for each business.
    """
    base_window = Window.partitionBy(group_col).orderBy(order_col) \
                        .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

    for l in lags:
        colname = f"rev_lag{l}"
        first_col = f"{colname}_first"
        
        # compute business-specific mean and fill nulls
        data = (
            data
            .withColumn(first_col, f.first(col(colname), ignorenulls=True).over(base_window))
            .withColumn(
                colname,
                f.when(col(colname).isNull(), col(first_col)).otherwise(col(colname))
            )
            .drop(first_col)
        )

    return data

def create_model(data, time_length, type):
    
    """
    Creates a GBT model for predicting future revenue and growth
    """

    lags=[1,3,6,12]
    required_cols = [f"{type}_{time_length}m"] #+ [f"rev_lag{l}" for l in lags]
    data=data.na.drop(subset=required_cols)
    data=impute_rev_lags_by_business(data, lags, 'merchant_abn', 'year_month')
    data=data.na.fill(0)
    pipeline = create_pl(lags, time_length, type)
    train, test = data.randomSplit([0.8,0.2], seed=42)
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    return predictions
    
def performance(predictions, time_length, type):
    """
    Checks performance using rmse and mae
    """
    
    rmse_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="rmse"
    )

    mae_rev = RegressionEvaluator(
        labelCol=f"{type}_{time_length}m",
        predictionCol="prediction",
        metricName="mae"
    )


    rmse = rmse_rev.evaluate(predictions)
    mae = mae_rev.evaluate(predictions)

    
    ranked = (
        predictions.groupBy("merchant_abn")
        .agg(f.avg("prediction").alias(f"pred_{type}"))
        
    )

    window = Window.partitionBy(f.lit(1)).orderBy(f.desc(f"pred_{type}"))
    
    # Add ranking column
    ranked = ranked.withColumn(f"{type}_rank", f.row_number().over(window))
    #print(f"RMSE: {rmse:.4f}")
    return mae, rmse, ranked

def combine(growth, revenue):
    """
    Combines ranking dfs into a composite ranking    
    """
    g=0.2
    rev=0.8
    combined = growth.join(revenue,
        on="merchant_abn",
        how="inner"  # or "outer" if some merchants are missing in one DF
        )

    # Create a composite rank (sum of ranks, lower is better)
    combined = (combined.withColumn(
            "composite_rank",
            f.round(g*f.col("growth_rank") + rev*f.col("rev_future_rank"),4)
            )
        .orderBy('composite_rank', ascending=True)
        )

    # Optionally, sort by the composite rank
    combined = combined.orderBy(f.asc("composite_rank"))
    return combined
    

In [54]:
merchant_transactions=spark.read.parquet('../data/curated/merchant_transactions')

In [55]:
merchant_abn_name=merchant_transactions.groupBy('merchant_abn', 'business').count()
merchant_abn_name

merchant_abn,business,count
17663034743,Semper Incorporated,1226
14878918457,Ultrices Limited,441
30389290864,Lobortis Industries,11415
16765088338,Duis Elementum Corp.,1660
82843556649,Tristique Pellent...,967
49505931725,Suspendisse Ac As...,67916
90058450104,Vel Turpis Consul...,164
19257772092,Dignissim Lacus C...,540
78774961599,Nulla Integer Lim...,624
22812667768,Feugiat Tellus Lo...,314


In [56]:
merchant_transactions=merchant_transactions.withColumn("year_month", f.date_format("order_datetime", "yyyy-MM"))
merchant_transactions=merchant_transactions.drop('user_id', 'business', 'order_datetime')

In [57]:
merchant_transactions

merchant_abn,dollar_value,biz_tags,rev_band,take_rate,segment,year_month
44160392990,83.98473054761176,digital goods: bo...,a,5.97,"Arts, Media & Ent...",2021-08
34082818630,28.990450388891105,digital goods: bo...,a,6.86,"Arts, Media & Ent...",2022-05
21025433654,23.188460173900875,digital goods: bo...,a,6.55,"Arts, Media & Ent...",2021-08
75034515922,23.232209957977723,digital goods: bo...,a,6.22,"Arts, Media & Ent...",2022-05
75034515922,20.319709965808922,digital goods: bo...,a,6.22,"Arts, Media & Ent...",2021-08
15560455575,274.97281365312296,digital goods: bo...,b,4.08,"Arts, Media & Ent...",2022-05
72472909171,73.00826801178104,digital goods: bo...,a,6.33,"Arts, Media & Ent...",2021-08
49505931725,39.74794708858456,digital goods: bo...,b,4.70,"Arts, Media & Ent...",2022-05
49505931725,149.6516401071932,digital goods: bo...,b,4.70,"Arts, Media & Ent...",2021-08
64974914166,98.52472498657673,digital goods: bo...,c,2.12,"Arts, Media & Ent...",2022-05


In [58]:
month_agg=(merchant_transactions.groupBy('merchant_abn','year_month', 'biz_tags', 'rev_band')
                                .agg(f.round(f.mean('take_rate'),4).alias('ave_take_rate'),
                                     f.round(f.sum('dollar_value'),4).alias('revenue'),
                                     (f.round(col('revenue')*col('ave_take_rate'),4)/100).alias('taking'))
)

In [59]:
month_agg=month_agg.orderBy('merchant_abn', 'year_month')
month_agg

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,1.26282
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,44.341822
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,49.720217000000005
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,54.20139
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,51.822782
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,52.975317
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,57.81328499999999
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,62.56039799999999
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,71.98792499999999
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,85.579887


In [60]:
taking=month_agg.groupBy('merchant_abn').agg(
                    f.sum('taking').alias('total_taking')
)

window = Window.partitionBy(f.lit(1)).orderBy(f.desc('total_taking'))
taking=taking.withColumn('rank', f.row_number().over(window))
taking

merchant_abn,total_taking,rank
32361057556,615238.464441,1
86578477987,613410.6043110001,2
45629217853,585698.2169259998,3
96680767841,577532.773368,4
21439773999,573644.5626579999,5
64403598239,559913.716104,6
82368304209,540589.3861499999,7
89726005175,532913.665932,8
94493496784,511093.37113,9
49322182190,498791.490643,10


In [61]:
window = Window.partitionBy("merchant_abn").orderBy("year_month")
lags = [3,6,12]
new_data=month_agg.withColumn(f"rev_lag1", f.lag("revenue", 1).over(window))
for lag in lags:
    new_data = new_data.withColumn(f"rev_lag{lag}", f.lag("revenue", lag).over(window))

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,1.26282,NULL,NULL,NULL,NULL
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,44.341822,701.5666,NULL,NULL,NULL
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,49.720217000000005,24634.3455,NULL,NULL,NULL
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,54.20139,27622.3428,701.5666,NULL,NULL
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,51.822782,30111.8833,24634.3455,NULL,NULL
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,52.975317,28790.4343,27622.3428,NULL,NULL
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,57.81328499999999,29430.7319,30111.8833,701.5666,NULL
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,62.56039799999999,32118.4915,28790.4343,24634.3455,NULL
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,71.98792499999999,34755.7764,29430.7319,27622.3428,NULL
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,85.579887,39993.2919,32118.4915,30111.8833,NULL


In [ ]:
for window_size in [3,6,12]:
    roll_window = Window.partitionBy("merchant_abn").orderBy("year_month").rowsBetween(-window_size+1,0)
    new_data = (
        new_data
        .withColumn(f"rev_mean_{window_size}", f.mean("revenue").over(roll_window))
        .withColumn(f"rev_std_{window_size}", f.stddev("revenue").over(roll_window))
    )

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,1.26282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,44.341822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,49.720217000000005,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,54.20139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,51.822782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,52.975317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,57.81328499999999,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,62.56039799999999,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,71.98792499999999,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,85.579887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746


In [63]:
new_data = (
    new_data
    .withColumn("rev_future_1m", f.lead("revenue", 1).over(window))
    .withColumn("growth_1m", (f.col("rev_future_1m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_3m", f.lead("revenue", 3).over(window))
    .withColumn("growth_3m", (f.col("rev_future_3m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_6m", f.lead("revenue", 6).over(window))
    .withColumn("growth_6m", (f.col("rev_future_6m") - f.col("revenue")) / f.col("revenue"))
    .withColumn("rev_future_12m", f.lead("revenue", 12).over(window))
    .withColumn("growth_12m", (f.col("rev_future_12m") - f.col("revenue")) / f.col("revenue"))
)

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12,rev_future_1m,growth_1m,rev_future_3m,growth_3m,rev_future_6m,growth_6m,rev_future_12m,growth_12m
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,1.26282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL,24634.3455,34.113338491313584,30111.8833,41.92091912585349,32118.4915,44.78110118127061,30182.6491,42.021787382694676
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,44.341822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,27622.3428,0.12129395928136183,28790.4343,0.16871115167236742,34755.7764,0.41086664551327345,31514.377,0.27928614949400626
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,49.720217000000005,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,30111.8833,0.0901277823545077,29430.7319,0.06546834615346242,39993.2919,0.4478602408771786,36365.1443,0.3165119469880738
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,54.20139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089,28790.4343,-0.04388463474152...,32118.4915,0.06663841580443422,47544.3817,0.578924214946064,35454.8418,0.17743687589278084
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,51.822782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623,29430.7319,0.022239942382529396,34755.7764,0.20719875351098824,44094.0579,0.5315523705038377,35735.9922,0.24124533265550632
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,52.975317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783,32118.4915,0.09132493235752663,39993.2919,0.35889559375857716,29431.268,1.821565300593731E-5,40122.3353,0.36328024176660045
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,57.81328499999999,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075,34755.7764,0.0821111072417583,47544.3817,0.4802806570165351,30182.6491,-0.06027189664246...,36497.3556,0.1363346749955552
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,62.56039799999999,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193,39993.2919,0.15069482090464806,44094.0579,0.26868286274278125,31514.377,-0.09326217785196712,44448.763,0.27888850729284803
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,71.98792499999999,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942,47544.3817,0.1888089087260157,29431.268,-0.2640948868727657,36365.1443,-0.09071890378696226,36755.0168,-0.08097045644797245
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,85.579887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746,44094.0579,-0.07257058934473426,30182.6491,-0.36516896380208896,35454.8418,-0.25427904344794533,NULL,NULL


In [64]:
new_data = (
    new_data
    .withColumn("month", f.month("year_month"))
    .withColumn("sin_month", f.sin(2 * m.pi * col("month") / 12))
    .withColumn("cos_month", f.cos(2 * m.pi * col("month") / 12))
)

new_data

merchant_abn,year_month,biz_tags,rev_band,ave_take_rate,revenue,taking,rev_lag1,rev_lag3,rev_lag6,rev_lag12,rev_mean_3,rev_std_3,rev_mean_6,rev_std_6,rev_mean_12,rev_std_12,rev_future_1m,growth_1m,rev_future_3m,growth_3m,rev_future_6m,growth_6m,rev_future_12m,growth_12m,month,sin_month,cos_month
10023283211,2021-02,"furniture, home f...",e,0.18,701.5666,1.26282,NULL,NULL,NULL,NULL,701.5666,NULL,701.5666,NULL,701.5666,NULL,24634.3455,34.113338491313584,30111.8833,41.92091912585349,32118.4915,44.78110118127061,30182.6491,42.021787382694676,2,0.8660254037844386,0.5000000000000001
10023283211,2021-03,"furniture, home f...",e,0.18,24634.3455,44.341822,701.5666,NULL,NULL,NULL,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,12667.956049999999,16923.030252828325,27622.3428,0.12129395928136183,28790.4343,0.16871115167236742,34755.7764,0.41086664551327345,31514.377,0.27928614949400626,3,1.0,6.123233995736766...
10023283211,2021-04,"furniture, home f...",e,0.18,27622.3428,49.720217000000005,24634.3455,NULL,NULL,NULL,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,17652.751633333333,14755.983108282884,30111.8833,0.0901277823545077,29430.7319,0.06546834615346242,39993.2919,0.4478602408771786,36365.1443,0.3165119469880738,4,0.8660254037844387,-0.4999999999999998
10023283211,2021-05,"furniture, home f...",e,0.18,30111.8833,54.20139,27622.3428,701.5666,NULL,NULL,27456.19053333333,2742.5462656801387,20767.53455,13563.437941753089,20767.53455,13563.437941753089,28790.4343,-0.04388463474152...,32118.4915,0.06663841580443422,47544.3817,0.578924214946064,35454.8418,0.17743687589278084,5,0.49999999999999994,-0.8660254037844387
10023283211,2021-06,"furniture, home f...",e,0.18,28790.4343,51.822782,30111.8833,24634.3455,NULL,NULL,28841.553466666664,1245.557245647741,22372.114500000003,12282.040574739623,22372.114500000003,12282.040574739623,29430.7319,0.022239942382529396,34755.7764,0.20719875351098824,44094.0579,0.5315523705038377,35735.9922,0.24124533265550632,6,1.224646799147353...,-1.0
10023283211,2021-07,"furniture, home f...",e,0.18,29430.7319,52.975317,28790.4343,27622.3428,NULL,NULL,29444.34983333333,660.8297443225855,23548.550733333337,11357.060790989783,23548.550733333337,11357.060790989783,32118.4915,0.09132493235752663,39993.2919,0.35889559375857716,29431.268,1.821565300593731E-5,40122.3353,0.36328024176660045,7,-0.4999999999999997,-0.8660254037844388
10023283211,2021-08,"furniture, home f...",e,0.18,32118.4915,57.81328499999999,29430.7319,30111.8833,701.5666,NULL,30113.219233333333,1765.8802059421507,28784.704883333336,2524.657921826413,24772.827985714288,10861.752853173075,34755.7764,0.0821111072417583,47544.3817,0.4802806570165351,30182.6491,-0.06027189664246...,36497.3556,0.1363346749955552,8,-0.8660254037844385,-0.5000000000000004
10023283211,2021-09,"furniture, home f...",e,0.18,34755.7764,62.56039799999999,32118.4915,28790.4343,24634.3455,NULL,32101.6666,2662.5621194049304,30471.61003333333,2577.743441343612,26020.696537500004,10657.444761577193,39993.2919,0.15069482090464806,44094.0579,0.26868286274278125,31514.377,-0.09326217785196712,44448.763,0.27888850729284803,9,-1.0,-1.83697019872102...
10023283211,2021-10,"furniture, home f...",e,0.18,39993.2919,71.98792499999999,34755.7764,29430.7319,27622.3428,NULL,35622.51993333334,4008.310566733992,32533.434883333335,4248.79224668721,27573.207133333333,11003.458096974942,47544.3817,0.1888089087260157,29431.268,-0.2640948868727657,36365.1443,-0.09071890378696226,36755.0168,-0.08097045644797245,10,-0.8660254037844386,0.5000000000000001
10023283211,2021-11,"furniture, home f...",e,0.18,47544.3817,85.579887,39993.2919,32118.4915,30111.8833,NULL,40764.48333333333,6429.0869141684525,35438.85128333333,7198.293176792175,29570.324589999997,12145.286021745746,44094.0579,-0.07257058934473426,30182.6491,-0.36516896380208896,35454.8418,-0.25427904344794533,NULL,NULL,11,-0.5000000000000004,0.8660254037844384


In [65]:
predictions_growth_1 = create_model(new_data, 1, 'growth')

In [66]:
mae_g_1, rmse_g_1, ranked_growth_1 = performance(predictions_growth_1, 1, 'growth')
mae_g_1, rmse_g_1

(1.592995658990947, 18.457630304381205)

In [67]:
predictions_rev_1=create_model(new_data, 1, 'rev_future')

In [68]:
mae_r_1, rsme_r_1, ranked_rev_1 = performance(predictions_rev_1, 1, 'rev_future')
mae_r_1, rsme_r_1


(3387.7939935222867, 9647.675125605503)

In [69]:
composite_rank_1m=combine(ranked_growth_1, ranked_rev_1)
window = Window.partitionBy(f.lit(1)).orderBy(f.asc("composite_rank"))
    
# Add ranking column
composite_rank_1m = composite_rank_1m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_1m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
79417999332,14.46339448217349,69,388003.74448297557,25,33.8
80518954462,9.06692467572404,146,360373.90868339804,33,55.6
45629217853,6.878741953625323,214,412825.87807026535,22,60.4
89726005175,9.596011320390765,138,336154.5750570062,45,63.6
38090089066,5.39610188664326,295,455447.6615148826,10,67.0
49322182190,6.815714426490591,217,378169.803497168,32,69.0
46804135891,7.1356184097872974,208,331749.10046993307,49,80.8
72472909171,7.237525618606765,206,314538.26550620364,53,83.6
57757792876,5.51461215007324,290,326947.66094295995,50,98.0
49505931725,5.691485710104394,286,302807.6808768316,57,102.8


In [70]:
predictions_rev_3 = create_model(new_data, 3,'rev_future')

In [71]:
mae_r_3, rmse_r_3, ranked_rev_3 = performance(predictions_rev_3, 3, 'rev_future')
mae_r_3, rmse_r_3

(3556.00649993208, 10868.932839164116)

In [72]:
predictions_growth_3 = create_model(new_data, 3, 'growth')

In [73]:
mae_g_3, rmse_g_3, ranked_growth_3 = performance(predictions_growth_3, 3, 'growth')
mae_g_3, rmse_g_3

(2.499995632814934, 49.97505917859258)

In [74]:
composite_rank_3m=combine(ranked_growth_3, ranked_rev_3)
#window = Window.partitionBy(f.lit(1)).orderBy(f.asc("composite_rank"))
    
# Add ranking column
composite_rank_3m = composite_rank_3m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_3m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
86578477987,11.355685482734607,168,484852.99343574326,7,39.2
64403598239,14.72471792177055,107,422013.77300292556,28,43.8
80518954462,31.536025146012324,44,319143.71796200285,50,48.8
88699453206,18.01349475679933,84,279758.6626444496,60,64.8
52959528548,8.553284136562612,233,425260.5948452001,24,65.8
50315283629,7.729801853609107,258,422400.09142903716,27,73.2
96680767841,6.352261595006356,331,449736.87859779317,14,77.4
66228393506,15.442057063551415,101,218350.38850253375,74,79.4
93558142492,7.260362367610052,282,358561.0910195721,39,87.6
45433476494,7.088080297789963,296,290651.31175825367,57,104.8


In [75]:
predictions_rev_6 = create_model(new_data, 6, 'rev_future')

In [76]:
mae_r_6, rmse_r_6, ranked_rev_6=performance(predictions_rev_6, 6, 'rev_future')
mae_r_6, rmse_r_6

(3782.7252017412457, 10467.739667694592)

In [77]:
predictions_growth_6 = create_model(new_data, 6, 'growth')

In [78]:
mae_g_6, rmse_g_6, ranked_growth_6 = performance(predictions_growth_6, 6, 'growth')
mae_g_6, rmse_g_6

(2.9908772890789903, 46.98725999208743)

In [79]:
composite_rank_6m=combine(ranked_growth_6, ranked_rev_6)
composite_rank_6m = composite_rank_6m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_6m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
96680767841,17.21700943052613,133,446778.8108418768,24,45.8
48534649627,17.361908069719068,131,414146.01708634134,31,51.0
68559320474,17.08246280069813,136,353057.3073906043,46,64.0
88699453206,21.976395589687744,91,266895.73560781474,62,67.8
27093785141,8.019210715076076,342,509040.18937223003,6,73.2
94493496784,9.195352930463633,292,454260.9062282484,19,73.6
76767266140,6.911897266026445,379,507426.45888451656,7,81.4
11439466003,17.610029450733883,127,222151.49116139294,73,83.8
64203420245,8.668991285144115,311,422643.1612637756,29,85.4
46804135891,8.3387080836614,322,403700.3895324502,33,90.8


In [80]:
predictions_rev_12 = create_model(new_data, 12, 'rev_future')

In [81]:
mae_r_12, rmse_r_12, ranked_rev_12 = performance(predictions_rev_12, 12, 'rev_future')
mae_r_12, rmse_r_12

(4346.221796745534, 13374.967712415897)

In [82]:
predictions_growth_12 = create_model(new_data, 12, 'growth')

In [83]:
mae_g_12, rmse_g_12, ranked_growth_12 = performance(predictions_growth_12, 12, 'growth')
mae_g_12, rmse_g_12

(2.8280382013186682, 22.834495505032336)

In [84]:
composite_rank_12m=combine(ranked_growth_12, ranked_rev_12)
composite_rank_12m = composite_rank_12m.withColumn("composite_rank_indx", f.row_number().over(window))
composite_rank_12m

merchant_abn,pred_growth,growth_rank,pred_rev_future,rev_future_rank,composite_rank
76767266140,13.196904990706193,186,426413.1314901156,21,54.0
49505931725,29.282170364671643,59,261573.72788550766,59,59.0
22033359776,24.892076438031854,76,296223.9034702161,56,60.0
49322182190,13.950183955528063,175,348160.5090200943,43,69.4
76818455401,32.32279089687413,43,147178.30201179528,83,75.0
84703983173,25.232116451862108,74,109279.28841353445,94,90.0
68559320474,9.293494771969714,295,361267.3747508807,40,91.0
88699453206,11.328149856641692,224,272730.87985807116,58,91.2
82368304209,17.03621848323167,124,146879.7212354122,84,92.0
48534649627,8.872482141053831,309,343351.9680318382,44,97.0


In [91]:
rank_1 = composite_rank_1m.withColumnRenamed("composite_rank_indx", "rank_1").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_3 = composite_rank_3m.withColumnRenamed("composite_rank_indx", "rank_3").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_6 = composite_rank_6m.withColumnRenamed("composite_rank_indx", "rank_6").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')
rank_12 = composite_rank_12m.withColumnRenamed("composite_rank_indx", "rank_12").drop('pred_growth', 'growth_rank', 'pred_rev_future', 'rev_future_rank', 'composite_rank')

# Join on merchant_abn
combined = reduce(lambda a, b: a.join(b, on="merchant_abn", how="outer"), [rank_1,rank_3,rank_6,rank_12])

# Compute average of available (non-null) ranks
combined = combined.withColumn(
    "ave_rank",
    (
        f.coalesce(col("rank_1"), f.lit(0)) +
        f.coalesce(col("rank_3"), f.lit(0)) +
        f.coalesce(col("rank_6"), f.lit(0)) +
        f.coalesce(col("rank_12"), f.lit(0))
    ) / (
        f.when(col("rank_1").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_3").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_6").isNotNull(), 1).otherwise(0) +
        f.when(col("rank_12").isNotNull(), 1).otherwise(0)
    )
).orderBy('ave_rank', ascending=True)

combined

merchant_abn,rank_1,rank_3,rank_6,rank_12,ave_rank
49505931725,102.8,368.2,218.4,59.0,187.1
49322182190,69.0,433.2,NULL,69.4,190.53333333333333
88699453206,635.2,64.8,67.8,91.2,214.75
79417999332,33.8,239.8,273.0,326.8,218.35000000000002
38918664617,NULL,229.4,NULL,NULL,229.4
78760357380,327.8,129.0,132.2,346.4,233.85
80518954462,55.6,48.8,557.6,316.6,244.65
93558142492,108.2,87.6,579.0,285.4,265.04999999999995
66228393506,129.2,79.4,372.8,506.2,271.9
72472909171,83.6,363.8,265.0,403.2,278.9
